# Chapter 3 / Paper 2

## Notebook 3: Empirical Analysis

This notebook documents sample integration, baseline associations, spatial lag and spatial error specifications, GEWI interaction and polarity analyses, interpolation diagnostics, and flexible-model checks.

> **Interpretation boundary.** The workflow is observational. Spatial parameters and flexible-model outputs are not interpreted as behavioral spillovers or causal treatment effects. Restricted empirical inputs are not included.

In [ ]:
from pathlib import Path

CHAPTER_DIR = Path.cwd().resolve()
if CHAPTER_DIR.name == 'notebooks':
    CHAPTER_DIR = CHAPTER_DIR.parent
if not (CHAPTER_DIR / 'data').exists():
    raise RuntimeError(
        'Start Jupyter from the chapter_3_paper_2 directory or its notebooks directory.'
    )

# Original analytical code below uses paths relative to the chapter directory.
import os
os.chdir(CHAPTER_DIR)


In [ ]:
## BLOCK 0 — Setup and Paths

In [ ]:
#  Title: Setup & Imports for Baseline Mobility–Sales Association

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# For spatial models
import libpysal
import esda
import statsmodels.api as sm
import statsmodels.formula.api as smf

# For warnings and aesthetics
import warnings
warnings.filterwarnings("ignore")

# Configure plotting style
plt.style.use("seaborn-whitegrid")
sns.set_palette("Set2")

# Paths to processed datasets
mobility_path = "data/private/mobility/aggregated_aug2024/mobility_by_sector.csv.gz"
sales_path = "data/private/sales/sales_aug2024_with_geo.csv.gz"
geometry_path = "data/private/census_tracts/BR_setores_2022.shp"

print(" Environment ready.")

In [ ]:
## Block 1 – Load All Datasets & Prepare Spatial Structure (for H1)

In [ ]:
#  Title: Load Sales, Mobility, and Census Geometry Data
import pandas as pd
from shapely.geometry import Point
import pyogrio
import os

# Paths
sales_path = "data/private/sales/sales_aug2024_with_geo.csv.gz"
mobility_path = "data/private/mobility/aggregated_aug2024/mobility_by_sector.csv.gz"
shapefile_path = "data/private/census_tracts/BR_setores_CD2022.shp"

#  Load Sales Data
print(" Loading sales data...")
sales_df = pd.read_csv(sales_path)
print(f" Sales data loaded: {sales_df.shape}")

#  Load Mobility Data
print(" Loading mobility data...")
mobility_df = pd.read_csv(mobility_path)
print(f" Mobility data loaded: {mobility_df.shape}")

#  Load Census Sector Shapefile (set CRS explicitly)
print(" Loading census sectors shapefile...")
setores_df = pyogrio.read_dataframe(shapefile_path, columns=["CD_GEOCODI", "geometry"])
setores_df.crs = "EPSG:4674"  # Explicitly set SIRGAS 2000 CRS
setores_df = setores_df.to_crs("EPSG:4326")  # Reproject to WGS84
print(f" Census sectors loaded: {setores_df.shape}")

In [ ]:
## Block 2 — Spatial Join (Sales + Mobility via Census Sector)

In [ ]:
#  Block 2 — Assign Census Sector to Sales Points (High Performance, Parallel)
#  Title: Spatial Join — Assign Census Sector to Sales Points via STRtree + Joblib

import pandas as pd
import pyogrio
from shapely.geometry import Point
from shapely.strtree import STRtree
from shapely.ops import transform
from functools import partial
from joblib import Parallel, delayed
from tqdm import tqdm
import pyproj
import time
import os

# Paths
sales_path = "data/private/sales/sales_aug2024_with_geo.csv.gz"
gpkg_path = "data/private/census_tracts/BR_setores_CD2022.gpkg"
gpkg_layer = "BR_setores_CD2022"
output_path = "data/private/sales/sales_aug2024_with_sector.csv.gz"

# Step 1: Load sales data
print(" Loading sales data...")
sales_df = pd.read_csv(sales_path)
print(f" Sales: {sales_df.shape}")

# Step 2: Create geometry column
print(" Creating geometry column...")
sales_df["geometry"] = sales_df.apply(lambda row: Point(row["Longitude"], row["Latitude"]), axis=1)

# Step 3: Load census sectors
print(" Loading census sectors...")
start = time.time()
setores_df = pyogrio.read_dataframe(gpkg_path, layer=gpkg_layer, columns=["CD_SETOR", "geometry"])
print(f" Loaded {setores_df.shape[0]:,} polygons in {time.time() - start:.2f}s")

# Step 4: CRS reproject (if needed)
if setores_df.crs is None or setores_df.crs.to_epsg() != 4326:
    print(" Reprojecting to EPSG:4326...")
    project = partial(
        pyproj.transform,
        pyproj.CRS("EPSG:4674"),
        pyproj.CRS("EPSG:4326")
    )
    setores_df["geometry"] = setores_df["geometry"].apply(lambda g: transform(project, g))
    print(" Reprojection complete.")

# Step 5: Spatial index setup
print(" Building spatial index...")
geoms = setores_df["geometry"].tolist()
codes = setores_df["CD_SETOR"].tolist()
index = STRtree(geoms)

# Step 6: Define lookup function
def match_sector(pt):
    matches = index.query(pt)
    for i in matches:
        if geoms[i].contains(pt):
            return codes[i]
    return None

# Step 7: Parallel spatial join
print(" Performing spatial join in parallel...")
start = time.time()
points = sales_df["geometry"].tolist()

with Parallel(n_jobs=-1, prefer="threads") as parallel:
    setores = parallel(delayed(match_sector)(pt) for pt in tqdm(points, desc=" Matching sectors"))

sales_df["CD_SETOR"] = setores
print(f" Spatial join done in {time.time() - start:.2f}s")
print(f" Points matched: {sales_df['CD_SETOR'].notna().sum():,} of {len(sales_df):,}")

# Step 8: Save result
sales_df.drop(columns=["geometry"], inplace=True)
sales_df.to_csv(output_path, index=False, compression="gzip")
print(f" Saved final file with census sectors: {output_path}")

In [ ]:
## Block 3 — Aggregate Sales and Merge with Mobility

In [ ]:
#  Inspect Columns of Mobility Dataset

import pandas as pd

# Caminho para o arquivo de mobilidade
mobility_path = "data/private/mobility/aggregated_aug2024/mobility_by_sector.csv.gz"

# Lê apenas o cabeçalho
df_mob = pd.read_csv(mobility_path, nrows=5)

print(" Columns in mobility dataset:")
for col in df_mob.columns:
    print(f"• {col}")
#  Preview formatted (Jupyter-friendly)
from IPython.display import display

mobility_df = pd.read_csv("data/private/mobility/aggregated_aug2024/mobility_by_sector.csv.gz")
display(mobility_df.head())


In [ ]:
import pandas as pd
import os
import time

# Paths
sales_path = "data/private/sales/sales_aug2024_with_sector.csv.gz"
mobility_path = "data/private/mobility/aggregated_aug2024/mobility_by_sector.csv.gz"
output_dir = "data/private/analysis"
output_path = os.path.join(output_dir, "sales_mobility_merged.csv.gz")

# Step 1: Load datasets
print(" Loading sales with census sector...")
sales_df = pd.read_csv(sales_path, usecols=["CD_SETOR", "total_sales"])
print(f" Sales data: {sales_df.shape}")

print(" Loading mobility data...")
mobility_df = pd.read_csv(mobility_path)
print(f" Mobility data: {mobility_df.shape}")

# Step 2: Aggregate sales by CD_SETOR
print(" Aggregating sales by CD_SETOR...")
start = time.time()
sales_agg = sales_df.groupby("CD_SETOR", as_index=False)["total_sales"].sum()
print(f" Aggregated: {sales_agg.shape} rows in {time.time()-start:.2f}s")

# Step 3: Merge with mobility data
print(" Merging with mobility data...")
start = time.time()
merged_df = pd.merge(
    sales_agg,
    mobility_df,
    left_on="CD_SETOR",
    right_on="code_censo",
    how="inner"
)
print(f" Merged dataset: {merged_df.shape} in {time.time()-start:.2f}s")

# Step 4: Save merged result
print(" Saving final merged file...")
os.makedirs(output_dir, exist_ok=True)
merged_df.to_csv(output_path, index=False, compression="gzip")
print(f" Done! Final file saved to:\n{output_path}")

In [ ]:
## Block 4 — Baseline association: Mobility → Sales

In [ ]:
#  Block 4 — Baseline association: Mobility → Sales
#  Title: Linear Regression — Mobility Impact on Sales (OLS, Real-time Output)

import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns
import time

# Step 1 — Load merged dataset
input_path = "data/private/analysis/sales_mobility_merged.csv.gz"
print(" Loading merged dataset...")
df = pd.read_csv(input_path)
print(f" Data loaded: {df.shape}")

# Step 2 — Prepare variables
print(" Preparing variables for regression...")
X = df["total_unique_visitors"]
y = df["total_sales"]

# Step 3 — Add constant and fit model
X = sm.add_constant(X)
print(" Fitting linear regression model (OLS)...")
start = time.time()
model = sm.OLS(y, X).fit()
print(f" Model fitted in {time.time()-start:.2f}s")

# Step 4 — Summary
print("\n Regression Results (H1):\n")
print(model.summary())

# Step 5 — Plot: Mobility vs Sales
plt.figure(figsize=(10, 6))
sns.scatterplot(x="total_unique_visitors", y="total_sales", data=df, alpha=0.5)
sns.regplot(x="total_unique_visitors", y="total_sales", data=df, scatter=False, color="red")
plt.title("H1 — Total Unique Visitors vs Total Sales")
plt.xlabel("Total Unique Visitors (Mobility)")
plt.ylabel("Total Sales (R$)")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
## Block 5A — Log Transformation + Robust OLS (H1 Refined)

In [ ]:
#  Title: Log-Transformed OLS Regression — Mobility → Log(Sales)

import pandas as pd
import statsmodels.api as sm
import numpy as np
import time

# Step 1: Load merged dataset
input_path = "data/private/analysis/sales_mobility_merged.csv.gz"
print(" Loading merged dataset...")
df = pd.read_csv(input_path)
print(f" Data loaded: {df.shape}")

# Step 2: Prepare variables
print(" Preparing variables (log transform)...")
df["log_sales"] = np.log1p(df["total_sales"])
X = sm.add_constant(df["total_unique_visitors"])
y = df["log_sales"]

# Step 3: Fit OLS model
print(" Fitting log-level regression model...")
start = time.time()
model = sm.OLS(y, X).fit()
elapsed = time.time() - start
print(f" Model fitted in {elapsed:.2f}s")

# Step 4: Display results
print("\n Regression Results (H1 - Refined with log_sales):\n")
print(model.summary())

In [ ]:
##Block 5B – Spatial Lag Model

In [ ]:
import sys


In [ ]:
## Block 6 — Spatial Lag Model (H1)

In [ ]:
#  Title: Spatial Lag Model — H1: Mobility → Sales (with spatial dependence)

import pandas as pd
from libpysal.weights import KNN
from spreg import ML_Lag
from sklearn.preprocessing import MinMaxScaler
import numpy as np

# Step 1: Load dataset
path = "data/private/analysis/sales_mobility_merged.csv.gz"
print(" Loading merged data...")
df = pd.read_csv(path)
print(f" Data loaded: {df.shape}")

# Step 2: Prepare variables
print(" Preparing variables...")
y = df["total_sales"].values.reshape(-1, 1)
X = df[["total_unique_visitors"]].values

# Step 3: Fix CD_SETOR and generate synthetic coordinates
df["CD_SETOR"] = df["CD_SETOR"].astype(str).str.replace(".0", "", regex=False).str.zfill(15)

scaler = MinMaxScaler()
df["x_coord"] = scaler.fit_transform(df["CD_SETOR"].str[-6:].astype(int).values.reshape(-1, 1))
df["y_coord"] = scaler.fit_transform(df["CD_SETOR"].str[:6].astype(int).values.reshape(-1, 1))

coords = df[["x_coord", "y_coord"]].values

# Step 4: Spatial Weights Matrix (KNN)
print(" Building spatial weights matrix...")
w = KNN.from_array(coords, k=8)
w.transform = "r"
print(" Spatial weights ready.")

# Step 5: Fit Spatial Lag Model
print(" Fitting Spatial Lag Model...")
model = ML_Lag(y, X, w=w, name_y="total_sales", name_x=["total_unique_visitors"], name_w="KNN-8")
print(" Model fitted.")

# Step 6: Display Results
print("\n Spatial Lag Model Results (H1):\n")
print(model.summary)

In [ ]:
##  Block 6 — Spatial Lag Model with Log-Transformed Sales

In [ ]:
#  Block 6 — Spatial Lag with Log Sales (Robust CD_SETOR Handling)
#  Title: Spatial Lag Regression (log_sales ~ mobility) with synthetic coords

import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from libpysal.weights import KNN
from spreg import ML_Lag
import time

# Step 1: Load merged dataset
print(" Loading merged data...")
df = pd.read_csv("data/private/analysis/sales_mobility_merged.csv.gz")
print(f" Data loaded: {df.shape}")

# Step 2: Prepare variables
print(" Preparing variables...")
df["log_sales"] = np.log1p(df["total_sales"])
df["CD_SETOR"] = df["CD_SETOR"].astype(str).str.replace(".0", "", regex=False).str.zfill(15)

# Step 3: Create synthetic spatial coordinates
scaler = MinMaxScaler()
df["x_coord"] = scaler.fit_transform(df["CD_SETOR"].str[-6:].astype(int).values.reshape(-1, 1))
df["y_coord"] = scaler.fit_transform(df["CD_SETOR"].str[:6].astype(int).values.reshape(-1, 1))

# Step 4: Create spatial weights matrix (KNN)
print(" Building spatial weights matrix...")
coords = df[["x_coord", "y_coord"]].values
w = KNN.from_array(coords, k=8)
w.transform = 'r'
print(" Spatial weights ready.")

# Step 5: Fit spatial lag model (log_sales ~ total_unique_visitors)
print(" Fitting Spatial Lag Model...")
y = df["log_sales"].values.reshape(-1, 1)
X = df[["total_unique_visitors"]].values
start = time.time()
model = ML_Lag(y, X, w=w, name_y="log_sales", name_x=["total_unique_visitors"])
print(f" Model fitted in {time.time() - start:.2f}s")

# Step 6: Show results
print("\n Spatial Lag Model Results (Log Sales):\n")
print(model.summary)

In [ ]:
## Block 6 — Spatial Error Model (log_sales)

In [ ]:
#  Block 6 — Spatial Error Model (H1)
#  Title: Spatial Error Model — log_sales ~ total_unique_visitors

import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from libpysal.weights import KNN
from spreg import GM_Error
import numpy as np
import time

# Step 1: Load dataset
print(" Loading merged dataset...")
df = pd.read_csv("data/private/analysis/sales_mobility_merged.csv.gz")
print(f" Data loaded: {df.shape}")

# Step 2: Log-transform sales
df["log_sales"] = np.log1p(df["total_sales"])

# Step 3: Generate synthetic spatial coordinates from CD_SETOR
scaler = MinMaxScaler()
df["x_coord"] = scaler.fit_transform(df["CD_SETOR"].astype(str).str[-6:].str.replace(".0", "", regex=False).astype(int).values.reshape(-1, 1))
df["y_coord"] = scaler.fit_transform(df["CD_SETOR"].astype(str).str[:6].astype(int).values.reshape(-1, 1))

# Step 4: Spatial weights matrix (KNN)
print(" Building spatial weights matrix (KNN=8)...")
coords = df[["x_coord", "y_coord"]].values
w = KNN.from_array(coords, k=8)
w.transform = 'r'
print(" Spatial weights ready.")

# Step 5: Prepare variables
y = df["log_sales"].values.reshape(-1, 1)
X = df[["total_unique_visitors"]].values

# Step 6: Fit Spatial Error Model
print(" Fitting Spatial Error Model (log_sales)...")
start = time.time()
model = GM_Error(y, X, w=w, name_y="log_sales", name_x=["total_unique_visitors"])
elapsed = time.time() - start
print(f" Model fitted in {elapsed:.2f}s\n")

# Step 7: Display results
print(" Spatial Error Model Results (H1):")
print(model.summary)

In [ ]:
#  Check available files in analysis folder
import os

path = "data/private/analysis/"
print(" Files in analysis:")
for file in sorted(os.listdir(path)):
    print("•", file)

In [ ]:
## Block 7 — Test GEWI interaction and polarity analyses with Interaction Effects (GEWI Moderation)

In [ ]:
#  Block 7 — Merge GEWI Data for Moderation Tests (GEWI interaction and polarity analyses)
#  Title: Merge GEWI by Sector into Sales-Mobility Dataset

import pandas as pd
import os

# Step 1: Load merged sales + mobility dataset
print(" Loading merged dataset...")
merged_path = "data/private/analysis/sales_mobility_merged.csv.gz"
df = pd.read_csv(merged_path)
print(f" Data loaded: {df.shape}")

# Step 2: Load GEWI data and merge by CD_SETOR
gewi_path = "data/private/gewi/gewis_por_setor.csv"
gewi_df = pd.read_csv(gewi_path)

# Sanitize column names
gewi_df.columns = gewi_df.columns.str.strip().str.lower()
df.columns = df.columns.str.strip().str.lower()

# Print available columns for manual verification
print("\n Columns in GEWI dataset:")
for col in gewi_df.columns:
    print(f"• {col}")

# Step 3: Merge GEWI into main dataset
print("\n Merging GEWI into dataset...")
merged_df = df.merge(gewi_df, how="left", left_on="cd_setor", right_on="cd_setor")
print(f" Final merged dataset: {merged_df.shape}")

# Step 4: Save to disk for future use
output_path = "data/private/analysis/sales_mobility_gewi_merged.csv.gz"
merged_df.to_csv(output_path, index=False, compression="gzip")
print(f" Final merged dataset with GEWI saved to:\n{output_path}")

In [ ]:
## Block 8 — GEWI interaction and polarity analyses: Interaction Effects with GEWI

In [ ]:
#  Title: Moderation Analysis — Mobility × GEWI (GEWI interaction and polarity analyses)

import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import time

# Paths
input_path = "data/private/analysis/sales_mobility_gewi_merged.csv.gz"

# Step 1: Load merged data
print(" Loading merged dataset...")
df = pd.read_csv(input_path)
print(f" Data loaded: {df.shape}")

# Step 2: Prepare variables
print(" Preparing variables for moderation...")
df = df.copy()
df["log_sales"] = np.log1p(df["total_sales"])  # Log transform
df["gewi_pos"] = df["gewi_v2_pos"]
df["gewi_neg"] = df["gewi_v2_neg"]

# Step 3: Interaction terms
df["mob_x_pos"] = df["total_unique_visitors"] * df["gewi_pos"]
df["mob_x_neg"] = df["total_unique_visitors"] * df["gewi_neg"]

# Step 4: Fit interaction model (mobility × GEWI)
print(" Fitting interaction model (GEWI pos & neg)...")
start = time.time()
model = smf.ols(
    formula="log_sales ~ total_unique_visitors + gewi_pos + gewi_neg + mob_x_pos + mob_x_neg",
    data=df
).fit()
print(f" Model fitted in {time.time() - start:.2f}s")

# Step 5: Display results
print("\n Regression Results (GEWI interaction and polarity analyses — Interaction Effects):\n")
print(model.summary())

In [ ]:
import pandas as pd

# Caminho para o arquivo
file_path = "data/private/analysis/sales_mobility_gewi_merged.csv.gz"

# Carregar apenas o cabeçalho e número de linhas
print(" Checking dataset structure...")
df = pd.read_csv(file_path)
print(f" Number of rows: {df.shape[0]:,}")
print(f" Number of columns: {df.shape[1]:,}")

# Mostrar nomes das colunas
print("\n Columns in dataset:")
for col in df.columns:
    print(f"• {col}")

In [ ]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("data/private/analysis/sales_mobility_gewi_merged.csv.gz")

# Create required variables
df["log_sales"] = np.log1p(df["total_sales"])
df["gewi_pos"] = df["gewi_v2_pos"]
df["gewi_neg"] = df["gewi_v2_neg"]
df["mob_x_pos"] = df["total_unique_visitors"] * df["gewi_pos"]
df["mob_x_neg"] = df["total_unique_visitors"] * df["gewi_neg"]

# Check for missing values
cols = ["total_unique_visitors", "gewi_pos", "gewi_neg", "mob_x_pos", "mob_x_neg", "log_sales"]
print(" Missing values per column:")
print(df[cols].isna().sum())

# Show number of valid rows
valid_rows = df[cols].dropna().shape[0]
print(f"\n Valid rows for regression: {valid_rows} of {len(df)}")

In [ ]:
import sys


In [ ]:
import pandas as pd

# Verifica as colunas do arquivo GEWI original
gewi_path = "data/private/gewi/gewis_por_setor.csv"
df = pd.read_csv(gewi_path, nrows=5)

print(" Columns in GEWI file:")
print(df.columns.tolist())

In [ ]:
## Full Code — Expand GEWI Coverage via Nearest Sector Assignment

In [ ]:
#  Title: Expand GEWI to All Census Sectors via Nearest Neighbor

import pandas as pd
import pyogrio
from shapely.geometry import shape
from shapely.strtree import STRtree
from shapely.ops import nearest_points
from tqdm import tqdm

# Paths
gewi_path = "data/private/gewi/gewis_por_setor.csv"
gpkg_path = "data/private/census_tracts/BR_setores_CD2022.gpkg"
output_path = "data/private/analysis/gewi_expanded_by_nearest.csv.gz"

# Step 1: Load GEWI data
print(" Loading GEWI data...")
gewi_df = pd.read_csv(gewi_path)
gewi_df["CD_SETOR"] = gewi_df["CD_SETOR"].astype(str)
print(f" GEWI loaded: {gewi_df.shape}")

# Step 2: Load census sector geometries
print(" Loading census sectors from GeoPackage...")
setores_df = pyogrio.read_dataframe(gpkg_path, layer="BR_setores_CD2022", columns=["CD_SETOR", "geometry"])
setores_df["CD_SETOR"] = setores_df["CD_SETOR"].astype(str)
print(f" Geometries loaded: {setores_df.shape}")

# Step 3: Identify missing GEWI sectors
merged_df = setores_df.merge(gewi_df, on="CD_SETOR", how="left")
missing_gewi_df = merged_df[merged_df["GEWI_v1"].isna()].copy()
print(f" Sectors with GEWI: {len(merged_df) - len(missing_gewi_df):,} — Missing: {len(missing_gewi_df):,}")

# Step 4: Build spatial index from GEWI-enabled geometries
print(" Building spatial index...")
gewi_geoms = merged_df[~merged_df["GEWI_v1"].isna()][["CD_SETOR", "geometry"]].copy()
geometries = list(gewi_geoms["geometry"])
geom_to_setor = dict(zip(geometries, gewi_geoms["CD_SETOR"]))
tree = STRtree(geometries)

# Step 5: Assign nearest GEWI sector
print(" Assigning nearest GEWI sector to missing sectors...")
tqdm.pandas()
missing_gewi_df["CD_SETOR_NEAREST"] = missing_gewi_df["geometry"].progress_apply(
    lambda geom: geom_to_setor.get(tree.nearest(geom))
)

# Step 6: Merge GEWI values from nearest neighbors
expanded_df = missing_gewi_df.merge(
    gewi_df,
    left_on="CD_SETOR_NEAREST",
    right_on="CD_SETOR",
    how="left",
    suffixes=("", "_from_nearest")
)

# Step 7: Prepare final GEWI for expanded set
gewi_df_clean = gewi_df[["CD_SETOR", "GEWI_v1", "GEWI_v2_pos", "GEWI_v2_neg", "GEWI_v3", "volume_tweets"]].copy()
expanded_df_clean = expanded_df[["CD_SETOR", "GEWI_v1", "GEWI_v2_pos", "GEWI_v2_neg", "GEWI_v3", "volume_tweets"]].copy()

# Reset index to avoid concat issues
gewi_df_clean = gewi_df_clean.reset_index(drop=True)
expanded_df_clean = expanded_df_clean.reset_index(drop=True)

# Step 8: Concatenate and remove duplicates
final_gewi = pd.concat([gewi_df_clean, expanded_df_clean], axis=0, ignore_index=True)
final_gewi = final_gewi.drop_duplicates(subset="CD_SETOR", keep="first")

# Step 9: Save final expanded GEWI
final_gewi.to_csv(output_path, index=False, compression="gzip")
print(f" Saved expanded GEWI file:\n{output_path}")

In [ ]:
## Block 9 — Merge Expanded GEWI with Sales + Mobility

In [ ]:
#  Block 9 — Merge Expanded GEWI with Sales + Mobility
#  Title: Final Merge — Sales, Mobility, and Expanded GEWI (for GEWI interaction and polarity analyses)

import pandas as pd
import time

# Paths
mobility_sales_path = "data/private/analysis/sales_mobility_merged.csv.gz"
gewi_expanded_path = "data/private/analysis/gewi_expanded_by_nearest.csv.gz"
output_path = "data/private/analysis/sales_mobility_gewi_expanded.csv.gz"

# Step 1: Load merged mobility + sales data
print(" Loading mobility + sales dataset...")
df = pd.read_csv(mobility_sales_path)
print(f" Loaded: {df.shape}")

# Step 2: Load expanded GEWI dataset
print(" Loading expanded GEWI data...")
gewi_df = pd.read_csv(gewi_expanded_path)
gewi_df.columns = gewi_df.columns.str.strip().str.lower()
gewi_df.rename(columns={"cd_setor": "code_censo"}, inplace=True)
print(f" GEWI Expanded: {gewi_df.shape}")

# Step 3: Merge on code_censo (CD_SETOR)
print(" Merging all datasets...")
start = time.time()
merged = df.merge(gewi_df, on="code_censo", how="inner")
print(f" Final merged dataset: {merged.shape} in {time.time()-start:.2f}s")

# Step 4: Save
merged.to_csv(output_path, index=False, compression="gzip")
print(f" Final dataset saved to:\n{output_path}")

In [ ]:
## Block 10 — GEWI interaction and polarity analyses — Re-Estimate Interaction Effects with Expanded GEWI

In [ ]:
#  Title: Spatial Join — Assign CD_SETOR to Sales Using lat/lon (no geopandas)
import pandas as pd
import pyogrio
import shapely
from shapely.geometry import Point, shape
from rtree import index
from tqdm import tqdm

# Paths
sales_path = "data/private/sales/sales_aug2024_with_geo.csv.gz"
gpkg_path = "data/private/census_tracts/BR_setores_CD2022.gpkg"
output_path = "data/private/sales/sales_with_cd_setor.csv.gz"

# Load sales data
print(" Loading sales data...")
df = pd.read_csv(sales_path)
df = df.dropna(subset=["Latitude", "Longitude"])
df["geometry"] = df.apply(lambda row: Point(row["Longitude"], row["Latitude"]), axis=1)
print(f" Loaded sales: {df.shape}")

# Load census sector geometries (using pyogrio instead of geopandas)
print(" Reading geometries from GPKG...")
records = pyogrio.read_dataframe(gpkg_path, layer="BR_setores_CD2022", columns=["CD_SETOR", "geometry"])
records["CD_SETOR"] = records["CD_SETOR"].astype(str)

# Build spatial index
print(" Building spatial index...")
spatial_index = index.Index()
geom_lookup = {}

for idx_val, row in records.iterrows():
    geom = row["geometry"]
    if geom is not None and not geom.is_empty:
        bounds = geom.bounds
        spatial_index.insert(idx_val, bounds)
        geom_lookup[idx_val] = (row["CD_SETOR"], geom)

# Assign CD_SETOR to each point
print(" Assigning CD_SETOR to points...")
cd_setor_list = []

for geom in tqdm(df["geometry"], total=len(df)):
    found = False
    for idx_candidate in spatial_index.intersection(geom.bounds):
        cd_setor, polygon = geom_lookup[idx_candidate]
        if polygon.contains(geom):
            cd_setor_list.append(cd_setor)
            found = True
            break
    if not found:
        cd_setor_list.append(None)

df["CD_SETOR"] = cd_setor_list
df = df.dropna(subset=["CD_SETOR"])
df["CD_SETOR"] = df["CD_SETOR"].astype(str)

# Save to disk
df.drop(columns=["geometry"]).to_csv(output_path, index=False, compression="gzip")
print(f" Saved output to:\n{output_path}")

In [ ]:
import pandas as pd

# Caminho do arquivo
file_path = "data/private/sales/sales_with_cd_setor.csv.gz"

# Carregar o arquivo
df = pd.read_csv(file_path, low_memory=False)

# Mostrar número de linhas e colunas
print(f" Loaded: {df.shape[0]} rows and {df.shape[1]} columns\n")

# Listar colunas
print(" Columns:")
print(df.columns.tolist())

# Mostrar 5 linhas aleatórias
print("\n Sample rows:")
print(df.sample(5, random_state=42))

In [ ]:
#  Block 10 — Test GEWI interaction and polarity analyses with Expanded GEWI (via CD_SETOR)
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import time

#  Paths
sales_path = "data/private/sales/sales_with_cd_setor.csv.gz"
mobility_path = "data/private/mobility/aggregated_aug2024/mobility_by_sector.csv.gz"
gewi_path = "data/private/analysis/gewi_with_coords.csv.gz"  # contém todos GEWIs

#  Load datasets
print(" Loading sales data...")
sales_df = pd.read_csv(sales_path, low_memory=False)
sales_df["CD_SETOR"] = sales_df["CD_SETOR"].astype(str)

print(" Loading mobility data...")
mob_df = pd.read_csv(mobility_path)
mob_df["code_censo"] = mob_df["code_censo"].astype(str)

print(" Loading GEWI data...")
gewi_df = pd.read_csv(gewi_path)
gewi_df["CD_SETOR"] = gewi_df["CD_SETOR"].astype(str)

#  Merge datasets
merged = sales_df.merge(mob_df, left_on="CD_SETOR", right_on="code_censo", how="inner")
print(f" Sales + Mobility merged: {merged.shape}")

merged = merged.merge(gewi_df.drop(columns=["lat", "lon", "ano_mes", "volume_tweets"]), on="CD_SETOR", how="inner")
print(f" Final merged dataset: {merged.shape}")

#  Save final dataset for modeling
output_path = "data/private/analysis/sales_mobility_gewi_expanded.csv.gz"
merged.to_csv(output_path, index=False, compression="gzip")
print(f" Final dataset saved to:\n{output_path}")

In [ ]:
##  Block 11 — GEWI interaction and polarity analyses Interaction Effects with GEWI (Expanded via IDW)

In [ ]:
#  Block 11 — GEWI interaction and polarity analyses Interaction Effects with GEWI (Expanded via IDW)
# Title: Test Interaction Effects (GEWI Moderation) — Expanded GEWI

import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import time

#  Step 1: Load merged dataset
path = "data/private/analysis/sales_mobility_gewi_expanded.csv.gz"
print(" Loading expanded dataset...")
df = pd.read_csv(path)
print(f" Data loaded: {df.shape}")

#  Step 2: Prepare variables
print(" Preparing variables...")
df["log_sales"] = np.log1p(df["total_sales"])
df["gewi_pos"] = df["GEWI_v2_pos"]
df["gewi_neg"] = df["GEWI_v2_neg"]
df["mob_x_pos"] = df["total_unique_visitors"] * df["gewi_pos"]
df["mob_x_neg"] = df["total_unique_visitors"] * df["gewi_neg"]

#  Step 3: Drop rows with missing values in key variables
df_clean = df.dropna(subset=[
    "log_sales", "total_unique_visitors", "gewi_pos", "gewi_neg",
    "mob_x_pos", "mob_x_neg"
])
print(f" Cleaned dataset for regression: {df_clean.shape[0]} rows")

#  Step 4: Fit OLS model
print(" Fitting interaction model (GEWI pos & neg)...")
start = time.time()
model = smf.ols(
    formula="log_sales ~ total_unique_visitors + gewi_pos + gewi_neg + mob_x_pos + mob_x_neg",
    data=df_clean
).fit()
print(f" Model fitted in {time.time() - start:.2f}s\n")

#  Step 5: Display results
print(" Regression Results (GEWI interaction and polarity analyses — Expanded GEWI):\n")
print(model.summary())

In [ ]:
## Block 12 — Visualizing GEWI Moderation Effect (Interaction Plots)

In [ ]:
# Title: Interaction Plot — Mobility × GEWI Positive (Tercis Only)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load data
path = "data/private/analysis/sales_mobility_gewi_expanded.csv.gz"
df = pd.read_csv(path)

# Prepare variables
df = df.copy()
df = df[df["GEWI_v2_pos"] > 0]  # Use only rows with positive GEWI
df["log_sales"] = np.log1p(df["total_sales"])
df["tercil_pos"] = pd.qcut(
    df["GEWI_v2_pos"],
    q=3,
    labels=["Low", "Medium", "High"],
    duplicates="drop"
)

# Plot settings
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df,
    x="total_unique_visitors",
    y="log_sales",
    hue="tercil_pos",
    palette="viridis",
    s=80,
    edgecolor="black"
)

# Add trend lines
sns.regplot(
    data=df[df["tercil_pos"] == "Low"],
    x="total_unique_visitors",
    y="log_sales",
    scatter=False,
    label="Low GEWI",
    color="blue"
)
sns.regplot(
    data=df[df["tercil_pos"] == "Medium"],
    x="total_unique_visitors",
    y="log_sales",
    scatter=False,
    label="Medium GEWI",
    color="green"
)
sns.regplot(
    data=df[df["tercil_pos"] == "High"],
    x="total_unique_visitors",
    y="log_sales",
    scatter=False,
    label="High GEWI",
    color="orange"
)

# Final touches
plt.title("Interaction: Mobility × GEWI Positive")
plt.xlabel("Total Unique Visitors")
plt.ylabel("Log Sales")
plt.legend(title="GEWI Pos (Tercis)")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
print(df["GEWI_v2_pos"].describe())
print("\nUnique values:\n", df["GEWI_v2_pos"].value_counts())

In [ ]:
df_filtered = df[
    (df["log_sales"] < 12) &
    (df["total_unique_visitors"] < 500000)
]

In [ ]:
# Valores observados: 0.349810 → 0.410035 → 0.720836
bins = [0.34, 0.41, 0.55, 0.73]
labels = ["Low", "Medium", "High"]

df["tercil_pos"] = pd.cut(
    df["GEWI_v2_pos"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

In [ ]:
# Recriar tercis (com dados filtrados)
df_filtered["tercil_pos"] = pd.qcut(
    df_filtered["GEWI_v2_pos"],
    q=3,
    labels=["Low", "Medium", "High"],
    duplicates="drop"
)

# Replot
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df_filtered,
    x="total_unique_visitors",
    y="log_sales",
    hue="tercil_pos",
    palette="viridis",
    s=80,
    edgecolor="black"
)

# Trend lines
for tercil, color in zip(["Low", "Medium", "High"], ["blue", "green", "orange"]):
    subset = df_filtered[df_filtered["tercil_pos"] == tercil]
    if not subset.empty:
        sns.regplot(
            data=subset,
            x="total_unique_visitors",
            y="log_sales",
            scatter=False,
            label=f"{tercil} GEWI",
            color=color
        )

# Final plot settings
plt.title("Interaction: Mobility × GEWI Positive (Filtered)")
plt.xlabel("Total Unique Visitors")
plt.ylabel("Log Sales")
plt.legend(title="GEWI Pos (Tercis)")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
## Interaction Plot — Mobility × GEWI Negative (Filtered)

In [ ]:
# Title: Interaction Plot — Mobility × GEWI Negative (Filtered)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression

# Load final merged data (sales + mobility + GEWI)
df = pd.read_csv("data/private/analysis/sales_mobility_gewi_expanded.csv.gz")
df["log_sales"] = np.log1p(df["total_sales"])

# Manual tercis (from GEWI_v2_neg.describe() and value_counts())
low_thres = 0.365
high_thres = 0.72

df_filtered = df[df["GEWI_v2_neg"].notna()].copy()
df_filtered["tercil_neg"] = pd.cut(
    df_filtered["GEWI_v2_neg"],
    bins=[-np.inf, low_thres, high_thres, np.inf],
    labels=["Low", "Medium", "High"]
)

# Plot
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df_filtered,
    x="total_unique_visitors",
    y="log_sales",
    hue="tercil_neg",
    palette="Set2",
    alpha=0.8,
    s=60
)

# Add regression lines per tercil
colors = {"Low": "blue", "Medium": "green", "High": "orange"}

for tercil in ["Low", "Medium", "High"]:
    subset = df_filtered[df_filtered["tercil_neg"] == tercil]
    if len(subset) > 1:
        model = LinearRegression()
        X = subset[["total_unique_visitors"]]
        y = subset["log_sales"]
        model.fit(X, y)
        x_range = np.linspace(X.min().values[0], X.max().values[0], 100)
        y_pred = model.predict(x_range.reshape(-1, 1))
        plt.plot(x_range, y_pred, label=f"{tercil} GEWI", color=colors[tercil])

plt.title("Interaction: Mobility × GEWI Negative (Filtered)")
plt.xlabel("Total Unique Visitors")
plt.ylabel("Log Sales")
plt.legend(title="GEWI Neg (Tercis)")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Title: Robust Interaction Plot — Mobility × GEWI Negative (Manual Terciles Fixed)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression

# Load final dataset
df = pd.read_csv("data/private/analysis/sales_mobility_gewi_expanded.csv.gz")
df["log_sales"] = np.log1p(df["total_sales"])

# Manual tercis (ajustado com base nos valores únicos observados)
# Exemplo: use três thresholds bem separados com base nos percentis reais
neg_values = df["GEWI_v2_neg"].dropna()
q1 = neg_values.quantile(0.33)
q2 = neg_values.quantile(0.66)

# Evita bins duplicados forçando quebra mínima
if q1 == q2:
    q1 = neg_values.min() + 0.01
    q2 = neg_values.max() - 0.01

df_filtered = df[df["GEWI_v2_neg"].notna()].copy()
df_filtered["tercil_neg"] = pd.cut(
    df_filtered["GEWI_v2_neg"],
    bins=[-np.inf, q1, q2, np.inf],
    labels=["Low", "Medium", "High"]
)

# Plot
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df_filtered,
    x="total_unique_visitors",
    y="log_sales",
    hue="tercil_neg",
    palette="Set2",
    alpha=0.8,
    s=60
)

# Add regression lines per tercil
colors = {"Low": "blue", "Medium": "green", "High": "orange"}

for tercil in ["Low", "Medium", "High"]:
    subset = df_filtered[df_filtered["tercil_neg"] == tercil]
    if len(subset) > 1:
        model = LinearRegression()
        X = subset[["total_unique_visitors"]]
        y = subset["log_sales"]
        model.fit(X, y)
        x_range = np.linspace(X.min().values[0], X.max().values[0], 100)
        y_pred = model.predict(x_range.reshape(-1, 1))
        plt.plot(x_range, y_pred, label=f"{tercil} GEWI", color=colors[tercil])

plt.title("Interaction: Mobility × GEWI Negative (Corrected Tercis)")
plt.xlabel("Total Unique Visitors")
plt.ylabel("Log Sales")
plt.legend(title="GEWI Neg (Tercis)")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
#### Block 13 — Robustness & Validation: VIF, Collinearity, Interpolation Comparison

In [ ]:
#  Title: Inspect Columns in Final Merged Dataset (GEWI + Mobility + Sales)

import pandas as pd

# File path
gewi_path = "data/private/analysis/sales_mobility_gewi_expanded.csv.gz"

# Load dataset
print(" Loading merged dataset...")
df = pd.read_csv(gewi_path, low_memory=False)
print(f" Loaded: {df.shape[0]:,} rows and {df.shape[1]} columns\n")

# Show all column names
print(" Columns in dataset:")
for col in df.columns:
    print(f"• {col}")

# Show 5 sample rows
print("\n Sample rows:")
display(df.sample(5))

In [ ]:
#  Block 13 — Robustness & Validation: VIF, Collinearity, Interpolation Comparison

import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import seaborn as sns

#  Load dataset
path = "data/private/analysis/sales_mobility_gewi_expanded.csv.gz"
df = pd.read_csv(path)
print(f" Dataset loaded: {df.shape}")

#  Prepare data
df = df.copy()
df = df.dropna(subset=["total_sales", "total_unique_visitors", "GEWI_v2_pos", "GEWI_v2_neg"])
df["log_sales"] = np.log1p(df["total_sales"])
df["mob_x_pos"] = df["total_unique_visitors"] * df["GEWI_v2_pos"]
df["mob_x_neg"] = df["total_unique_visitors"] * df["GEWI_v2_neg"]

#  VIF Calculation
X = df[["total_unique_visitors", "GEWI_v2_pos", "GEWI_v2_neg", "mob_x_pos", "mob_x_neg"]]
X = sm.add_constant(X)

vif_data = pd.DataFrame()
vif_data["Variable"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

print("\n Variance Inflation Factor (VIF):")
display(vif_data)

#  Re-run regression for comparison
model = sm.OLS(df["log_sales"], X).fit()
print("\n Re-estimated Model Summary (with VIF variables):")
print(model.summary())

#  Plot correlation heatmap
plt.figure(figsize=(8,6))
sns.heatmap(df[["log_sales", "total_unique_visitors", "GEWI_v2_pos", "GEWI_v2_neg", "mob_x_pos", "mob_x_neg"]].corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()

In [ ]:
## BLOCK S1 — Real centroids + spatial weights + diagnostics

In [ ]:
# --- BLOCK S1_ALT: Load BR census sectors from 10 GeoJSON parts (no fiona) ---
# Goal: build a dataframe with CD_SETOR, geometry, lat, lon using only pyogrio + shapely

import os, glob
import pyogrio
import pandas as pd

# ⇩ update this to your folder that has the 10 files
PARTS_DIR = "data/private/census_tracts"  # <- you said the files are here

# We expect files like BR_Census_part_01.geojson ... BR_Census_part_10.geojson
GLOB_PATTERN = os.path.join(PARTS_DIR, "BR_Census_part_*.geojson")

# Acceptable sector-ID field names we might encounter
SECTOR_COL_CANDIDATES = ["CD_SETOR", "CD_GEOCODI", "CD_GEOCOD", "code_censo", "CD_GEOCODI"]

def pick_sector_col(cols):
    for c in SECTOR_COL_CANDIDATES:
        if c in cols:
            return c
    return None

def read_one(path):
    """
    Read a single GeoJSON part with pyogrio, detect the sector column,
    then re-read only [sector_col, geometry] to save memory.
    """
    # Probe a tiny sample to discover columns
    head = pyogrio.read_dataframe(path, max_features=5)
    sector_col = pick_sector_col(head.columns)
    if sector_col is None:
        raise ValueError(
            f"Could not find sector ID column in {os.path.basename(path)}. "
            f"Looked for {SECTOR_COL_CANDIDATES}. Found: {list(head.columns)}"
        )

    # Now read just the two necessary columns
    g = pyogrio.read_dataframe(path, columns=[sector_col, "geometry"])
    # Some GeoJSONs ship without explicit CRS; assume WGS84 if missing
    if g.crs is None:
        g.set_crs("EPSG:4326", inplace=True)

    g.rename(columns={sector_col: "CD_SETOR"}, inplace=True)
    g["CD_SETOR"] = g["CD_SETOR"].astype(str)
    return g

# Collect and read all parts
paths = sorted(glob.glob(GLOB_PATTERN))
if not paths:
    raise FileNotFoundError(f"No files matched {GLOB_PATTERN}. Check PARTS_DIR and names.")

print(" Will read the following GeoJSON parts:")
for p in paths:
    print(" •", p)

parts = []
for p in paths:
    print(f" Reading {os.path.basename(p)} ...")
    g = read_one(p)
    print(f"   → {len(g):,} polygons")
    parts.append(g[["CD_SETOR", "geometry"]])

# Concatenate all
g_all = pd.concat(parts, ignore_index=True)
# Drop exact duplicates if any (some splits overlap at boundaries in rare cases)
g_all = g_all.drop_duplicates(subset=["CD_SETOR"]).reset_index(drop=True)

# Build centroids (Shapely 2.0)
g_all["lat"] = g_all.geometry.centroid.y
g_all["lon"] = g_all.geometry.centroid.x

coords = g_all[["CD_SETOR", "lat", "lon"]].copy()

print(f"\n Loaded sectors from GeoJSON parts: {len(g_all):,} unique polygons")
print(f" coords ready: {coords.shape} with columns {list(coords.columns)}")
# If later you need polygon adjacency (Queen/Rook), keep g_all around for weights construction.

In [ ]:
## S2 — Build Spatial Weights (KNN + optional Queen for the analysis subset)

In [ ]:
# --- BLOCK S2 (robust): Spatial weights & Moran’s I (Observed-only vs Interpolated-IDW2km) ---
# English-only code. No Fiona. Uses: pandas, numpy, pyogrio, shapely (via GeoDataFrame), libpysal, esda

import os, glob
import pandas as pd
import numpy as np
import pyogrio
from libpysal.weights import KNN
from esda import Moran

# ---------------- PATHS ----------------
PARTS_DIR   = "data/private/census_tracts"  # where BR_Census_part_*.geojson live
PARTS_GLOB  = os.path.join(PARTS_DIR, "BR_Census_part_*.geojson")

# Observed-only (preferred pre-merged file; if not present, we rebuild from sales + mobility)
OBSERVED_MERGED = "data/private/analysis/sales_mobility_merged.csv.gz"

# If we need to rebuild observed from raw components:
SALES_WITH_SETOR = "data/private/sales/sales_with_cd_setor.csv.gz"
MOBILITY         = "data/private/mobility/aggregated_aug2024/mobility_by_sector.csv.gz"

# Interpolated (IDW 2km) merged dataset (should contain lat/lon; if not, we'll join from coords)
INTERP_2KM       = "data/private/analysis/final_merged_interpolated.csv.gz"

# ---------------- HELPERS ----------------
SECTOR_COL_CANDIDATES = ["CD_SETOR", "CD_GEOCODI", "CD_GEOCOD", "code_censo"]

def pick_sector_col(cols):
    for c in SECTOR_COL_CANDIDATES:
        if c in cols:
            return c
    return None

def enforce_sector_key(df, colname):
    df[colname] = (
        df[colname].astype(str)
                   .str.replace(".0", "", regex=False)
                   .str.zfill(15)
    )
    return df

def load_coords_from_geojson(parts_glob):
    """Read all GeoJSON parts via pyogrio, compute centroids in projected CRS, return DataFrame indexed by CD_SETOR with lat/lon."""
    paths = sorted(glob.glob(parts_glob))
    if not paths:
        raise FileNotFoundError(f"No files matched {parts_glob}")
    frames = []
    print(" Loading centroids from GeoJSON parts (pyogrio, projected centroids)…")
    for p in paths:
        head = pyogrio.read_dataframe(p, max_features=5)
        sec_col = pick_sector_col(head.columns)
        if not sec_col:
            raise ValueError(f"Sector ID column not found in {os.path.basename(p)}; saw {list(head.columns)}")
        g = pyogrio.read_dataframe(p, columns=[sec_col, "geometry"])
        g = g.rename(columns={sec_col: "CD_SETOR"})
        g["CD_SETOR"] = g["CD_SETOR"].astype(str)
        if g.crs is None:
            g.set_crs("EPSG:4326", inplace=True)

        # Compute centroids in projected CRS to avoid warnings; EPSG:3857 is fine for centroiding
        g_proj = g.to_crs("EPSG:3857")
        cent = g_proj.geometry.centroid
        cent_wgs84 = cent.to_crs("EPSG:4326")

        out = pd.DataFrame({
            "CD_SETOR": g["CD_SETOR"].values,
            "lat": cent_wgs84.y.values,
            "lon": cent_wgs84.x.values
        })
        frames.append(out)

    coords = pd.concat(frames, ignore_index=True).drop_duplicates("CD_SETOR")
    coords = coords.set_index("CD_SETOR")
    print(f" coords ready: {coords.shape} (index=CD_SETOR)")
    return coords

def tiny_safe_k(n, default_k=8):
    """Pick a safe k for KNN given n (avoid k>=n)."""
    return int(max(1, min(default_k, n - 1)))

def build_knn_w(df_indexed, k):
    """df_indexed must be indexed by CD_SETOR and have lat/lon columns."""
    arr = df_indexed[["lat", "lon"]].to_numpy()
    ids = df_indexed.index.astype(str).tolist()
    w = KNN.from_array(arr, k=k, ids=ids)
    w.transform = "r"
    return w

def moran_report(y, w, label):
    mi = Moran(y, w, two_tailed=True)
    print(f"{label}: I={mi.I:.4f}, p={mi.p_sim:.4f}, EI={mi.EI:.4f}")

# ---------------- LOAD COORDS ----------------
coords = load_coords_from_geojson(PARTS_GLOB)

# ---------------- OBSERVED-ONLY SAMPLE ----------------
print("\n Building observed-only sample (Sales + Mobility; no GEWI required here)…")
if os.path.exists(OBSERVED_MERGED):
    obs = pd.read_csv(OBSERVED_MERGED, low_memory=False)
    # Expect columns: CD_SETOR, total_sales, mobility metrics, etc.
    enforce_sector_key(obs, "CD_SETOR")
    # Keep one row per sector, sum sales just in case:
    obs = (obs.groupby("CD_SETOR", as_index=False)["total_sales"]
              .sum()
              .rename(columns={"total_sales": "sales_obs"}))
    print(f" observed source: {OBSERVED_MERGED} (rows={len(obs)})")
else:
    print(" observed merged not found → rebuilding from SALES + MOBILITY")
    sales = pd.read_csv(SALES_WITH_SETOR, low_memory=False)
    enforce_sector_key(sales, "CD_SETOR")
    sales_agg = sales.groupby("CD_SETOR", as_index=False)["total_sales"].sum().rename(columns={"total_sales": "sales_obs"})

    mob = pd.read_csv(MOBILITY, low_memory=False)
    enforce_sector_key(mob, "code_censo")

    obs = sales_agg.merge(mob[["code_censo"]], left_on="CD_SETOR", right_on="code_censo", how="inner").drop(columns=["code_censo"])
    print(f" observed rebuilt: rows={len(obs)}")

# Join lat/lon
obs = obs.join(coords, on="CD_SETOR", how="inner")
obs = obs.dropna(subset=["lat", "lon"]).drop_duplicates("CD_SETOR").set_index("CD_SETOR")
obs["log_sales"] = np.log1p(obs["sales_obs"])
print(f"[OBS] rows={len(obs)}, unique sectors={obs.index.nunique()}, lat/lon finite? {np.isfinite(obs[['lat','lon']].to_numpy()).all()}")

# ---------------- INTERPOLATED SAMPLE (IDW 2km) ----------------
print("\n Loading interpolated (IDW 2km) sample…")
int1 = pd.read_csv(INTERP_2KM, low_memory=False)
enforce_sector_key(int1, "CD_SETOR")

# ensure lat/lon present
if not {"lat", "lon"}.issubset(int1.columns):
    int1 = int1.join(coords, on="CD_SETOR", how="left")

int1 = int1[["CD_SETOR", "lat", "lon", "total_sales_sum"]].drop_duplicates("CD_SETOR")
int1 = int1.dropna(subset=["lat","lon"])
int1 = int1.set_index("CD_SETOR")
int1["log_sales"] = np.log1p(int1["total_sales_sum"])
print(f"[INT] rows={len(int1)}, unique sectors={int1.index.nunique()}, lat/lon finite? {np.isfinite(int1[['lat','lon']].to_numpy()).all()}")

# ---------------- WEIGHTS (KNN) ----------------
print("\n Building KNN weights…")
k_obs = tiny_safe_k(len(obs), 8)
k_int = tiny_safe_k(len(int1), 8)

W_obs = build_knn_w(obs, k_obs) if len(obs) > 1 else None
W_int = build_knn_w(int1, k_int) if len(int1) > 1 else None

if W_obs:
    print(f"[OBS W] n={W_obs.n}, k={k_obs}, islands={len(W_obs.islands)}, components={W_obs.n_components}")
if W_int:
    print(f"[INT W] n={W_int.n}, k={k_int}, islands={len(W_int.islands)}, components={W_int.n_components}")

# ---------------- MORAN’S I (log_sales) ----------------
print("\n--- Moran’s I (log_sales) ---")
if W_obs:
    y_obs = obs.loc[W_obs.id_order, "log_sales"].to_numpy()
    moran_report(y_obs, W_obs, "OBSERVED")
else:
    print("OBSERVED: not enough data to build weights.")

if W_int:
    y_int = int1.loc[W_int.id_order, "log_sales"].to_numpy()
    moran_report(y_int, W_int, "INTERPOLATED")
else:
    print("INTERPOLATED: not enough data to build weights.")

print("\n BLOCK S2 complete.")

In [ ]:
## W1 (patched) — Build & Save Spatial Weights (Observed + Interpolated)

In [ ]:
# --- BLOCK W1: Build & Export Spatial Weights (OBS vs INT) -------------------
# Requirements: pyogrio, shapely, pandas, numpy, libpysal, esda, scipy, matplotlib

import os, glob
import numpy as np
import pandas as pd
import pyogrio
from shapely.geometry import Point
from libpysal.weights import KNN
from esda.moran import Moran
from scipy import sparse
import matplotlib.pyplot as plt

# ---------------------- CONFIG: paths & params -------------------------------
PARTS_DIR = "data/private/census_tracts"  # GeoJSON parts dir
GLOB_PATTERN = os.path.join(PARTS_DIR, "BR_Census_part_*.geojson")

OBS_PATH = "data/private/analysis/sales_mobility_merged.csv.gz"
INT_PATH = "data/private/analysis/final_merged_interpolated.csv.gz"

OUT_DIR = "data/private/analysis/weights"
os.makedirs(OUT_DIR, exist_ok=True)

K = 8  # KNN k
OBS_SALES_COL = "total_sales"       # in OBS_PATH
INT_SALES_COL = "total_sales_sum"   # in INT_PATH

# ---------------------- helpers ---------------------------------------------

def read_centroids_from_parts():
    """
    Read 10 GeoJSON parts (pyogrio only), compute centroids in a projected CRS
    to avoid warnings, then return DataFrame indexed by CD_SETOR with lat/lon.
    """
    paths = sorted(glob.glob(GLOB_PATTERN))
    if not paths:
        raise FileNotFoundError(f"No files matched: {GLOB_PATTERN}")

    def pick_sector_col(cols):
        for c in ["CD_SETOR", "CD_GEOCODI", "CD_GEOCOD", "code_censo", "CD_GEOCODI"]:
            if c in cols: return c
        return None

    frames = []
    print(" Loading centroids from GeoJSON parts (pyogrio, projected centroids)…")
    for p in paths:
        head = pyogrio.read_dataframe(p, max_features=5)
        sector_col = pick_sector_col(head.columns)
        if sector_col is None:
            raise ValueError(f"Could not find sector column in {os.path.basename(p)}; found {list(head.columns)}")

        g = pyogrio.read_dataframe(p, columns=[sector_col, "geometry"])
        # set CRS if missing; assume WGS84
        if g.crs is None:
            g.set_crs("EPSG:4326", inplace=True)

        # project to a metric CRS for centroid, then back to WGS84 for lat/lon
        try:
            g_m = g.to_crs("EPSG:3857")
        except Exception:
            # fallback to WGS84 if proj not available
            g_m = g

        g["CD_SETOR"] = g[sector_col].astype(str).str.replace(".0", "", regex=False).str.zfill(15)
        # centroids in projected CRS
        cent = g_m.geometry.centroid
        # back to WGS84
        try:
            cent_wgs = cent.to_crs("EPSG:4326")
            lat = cent_wgs.y
            lon = cent_wgs.x
        except Exception:
            # shapely arrays may not carry CRS; get x/y directly (WGS84)
            lat = cent.y
            lon = cent.x

        frames.append(pd.DataFrame({"CD_SETOR": g["CD_SETOR"], "lat": lat, "lon": lon}))

    df = pd.concat(frames, ignore_index=True)
    # keep first occurrence per sector
    df = df.drop_duplicates(subset="CD_SETOR").set_index("CD_SETOR")[["lat", "lon"]]
    print(f" coords ready: {df.shape} (index=CD_SETOR)")
    return df

def enforce_sector(df, col):
    """Standardize a sector key column to 15-char string named CD_SETOR."""
    df[col] = df[col].astype(str).str.replace(".0", "", regex=False).str.zfill(15)
    if "CD_SETOR" not in df.columns:
        df = df.rename(columns={col: "CD_SETOR"})
    elif col != "CD_SETOR":
        # keep canonical name
        df["CD_SETOR"] = df[col]
    return df

def safe_join_coords(df, coords, id_col="CD_SETOR", drop_dupes=True):
    """
    Join lat/lon from coords into df without clobbering existing lat/lon columns.
    """
    # If df already has lat/lon, rename them to avoid overlaps
    existing = []
    for c in ("lat", "lon"):
        if c in df.columns:
            df = df.rename(columns={c: f"{c}_orig"})
            existing.append(c)
    out = df.join(coords, on=id_col, how="inner")
    if drop_dupes:
        out = out.drop_duplicates(subset=[id_col])
    return out

def build_knn_w(df_xy, k=8):
    """Build row-standardized KNN weights from a DataFrame with lat/lon."""
    arr = df_xy[["lat", "lon"]].to_numpy()
    ids = df_xy["CD_SETOR"].tolist()
    w = KNN.from_array(arr, k=k, ids=ids)
    w.transform = "r"
    return w

def moran_diag(y, w, label):
    """Compute Moran's I and print concise diagnostics."""
    # libpysal expects y as 1D array aligned to w.id_order
    s = pd.Series(y, index=w.id_order).reindex(w.id_order)
    I = Moran(s.values, w)
    print(f"{label}: I={I.I:.4f}, p={I.p_norm:.4f}, EI={I.EI:.4f}")

def save_w_all(w, tag, out_dir=OUT_DIR):
    """Save a libpysal W as: CSR .npz, edge list .csv, and id_order .txt."""
    csr = w.sparse.tocsr()
    sparse.save_npz(os.path.join(out_dir, f"W_{tag}_knn{K}_rowstd.npz"), csr)

    rows = []
    for i_id in w.id_order:
        neighs = w.neighbors[i_id]
        wts = w.weights[i_id]
        rows.extend((i_id, j_id, wt) for j_id, wt in zip(neighs, wts))
    pd.DataFrame(rows, columns=["i","j","w"]).to_csv(
        os.path.join(out_dir, f"W_{tag}_edges_knn{K}_rowstd.csv"), index=False
    )

    with open(os.path.join(out_dir, f"W_{tag}_id_order.txt"), "w") as f:
        f.write("\n".join(map(str, w.id_order)))

    print(f" Saved W ({tag}) → .npz, .csv, id_order.txt")

def plot_sparsity(w, tag):
    """Optional: save a sparsity-pattern PNG for visual appendix."""
    csr = w.sparse.tocsr()
    plt.figure(figsize=(6,6), dpi=150)
    plt.spy(csr, markersize=0.5)
    plt.title(f"Sparsity pattern — W_{tag} (KNN={K}, row-std)")
    plt.tight_layout()
    png_path = os.path.join(OUT_DIR, f"W_{tag}_knn{K}_sparsity.png")
    plt.savefig(png_path, dpi=150)
    plt.close()
    print(f" Saved sparsity pattern: {png_path}")

# ---------------------- main flow -------------------------------------------

# 1) Load centroids
coords = read_centroids_from_parts()

# 2) Observed sample (no GEWI needed here — just Sales + Mobility)
print("\n Building observed-only sample…")
obs = pd.read_csv(OBS_PATH, low_memory=False)
# Normalize keys
if "CD_SETOR" not in obs.columns and "cd_setor" in obs.columns:
    obs = obs.rename(columns={"cd_setor": "CD_SETOR"})
enforce_sector(obs, "CD_SETOR")

# sanity for sales col
if OBS_SALES_COL not in obs.columns:
    raise KeyError(f"'{OBS_SALES_COL}' not found in {OBS_PATH}. Available: {list(obs.columns)}")

obs_xy = safe_join_coords(
    obs[["CD_SETOR", OBS_SALES_COL]].copy(), coords, id_col="CD_SETOR"
)
obs_xy = obs_xy[["CD_SETOR", "lat", "lon", OBS_SALES_COL]].dropna()
print(f"   [OBS] {obs_xy.shape}, unique sectors={obs_xy['CD_SETOR'].nunique()}")

# 3) Interpolated sample (IDW 2km)
print(" Loading interpolated (IDW 2km) sample…")
inter = pd.read_csv(INT_PATH, low_memory=False)

# Normalize CD_SETOR
if "CD_SETOR" not in inter.columns and "cd_setor" in inter.columns:
    inter = inter.rename(columns={"cd_setor": "CD_SETOR"})
enforce_sector(inter, "CD_SETOR")

if INT_SALES_COL not in inter.columns:
    raise KeyError(f"'{INT_SALES_COL}' not found in {INT_PATH}. Available: {list(inter.columns)}")

int_xy = safe_join_coords(
    inter[["CD_SETOR", INT_SALES_COL]].copy(), coords, id_col="CD_SETOR"
)
int_xy = int_xy[["CD_SETOR", "lat", "lon", INT_SALES_COL]].dropna()
print(f"   [INT] {int_xy.shape}, unique sectors={int_xy['CD_SETOR'].nunique()}")

# 4) Build KNN weights (row-standardized)
print("\n Building KNN weights…")
W_obs = build_knn_w(obs_xy[["CD_SETOR","lat","lon"]], k=K)
W_int = build_knn_w(int_xy[["CD_SETOR","lat","lon"]], k=K)

def w_quick_diag(w, name):
    islands = len(w.islands)
    comps = w.n_components
    print(f"[{name}] n={w.n}, islands={islands}, components={comps}")

w_quick_diag(W_obs, "OBS")
w_quick_diag(W_int, "INT")

# 5) Moran's I on log sales for each sample
print("\n--- Moran’s I (log sales) ---")
moran_diag(np.log1p(obs_xy[OBS_SALES_COL].values), W_obs, "OBSERVED")
moran_diag(np.log1p(int_xy[INT_SALES_COL].values), W_int, "INTERPOLATED")

# 6) Export weights and optional sparsity plots
print("\n Saving weights…")
save_w_all(W_obs, "OBS", OUT_DIR)
save_w_all(W_int, "INT", OUT_DIR)

# Optional visuals (handy for appendix/response)
try:
    plot_sparsity(W_obs, "OBS")
    plot_sparsity(W_int, "INT")
except Exception as e:
    print(f"(Skipping sparsity plots: {e})")

print("\n W1 finished.")

In [ ]:
##ONE-BLOCK: Recompute H1 tables with correct 95% CIs (no manual math)

In [ ]:
# --- ONE-BLOCK (FINAL): Recompute H1 tables with correct 95% CIs (robust/clustered) ---
# Inputs: data/private/analysis/sales_mobility_merged.csv.gz
# Outputs: data/private/analysis/tables/baseline_level_CIs.csv, baseline_log_CIs.csv

import os
import numpy as np
import pandas as pd
import statsmodels.api as sm

IN_PATH  = "data/private/analysis/sales_mobility_merged.csv.gz"
OUT_DIR  = "data/private/analysis/tables"
Y_LEVEL  = "total_sales"
Y_LOG    = "log_sales"
X_MAIN   = "total_unique_visitors"
GROUP_CL = "state"  # if missing in data, cluster spec is skipped

os.makedirs(OUT_DIR, exist_ok=True)

def _param_index(res):
    """Return an Index for parameters even if res.params is ndarray."""
    idx = getattr(res.params, "index", None)
    if idx is not None:
        return idx
    # fall back to model exog names; otherwise create generic names
    names = getattr(getattr(res, "model", None), "exog_names", None)
    if names is None:
        names = [f"b{i}" for i in range(len(res.params))]
    return pd.Index(names)

def _confint_df(res):
    """Return a CI DataFrame with proper index and ['CI_low','CI_high'] columns."""
    ci = res.conf_int(alpha=0.05)
    idx = _param_index(res)
    if isinstance(ci, np.ndarray):
        return pd.DataFrame(ci, index=idx, columns=["CI_low","CI_high"])
    ci = ci.copy()
    ci.columns = ["CI_low","CI_high"]
    # ensure alignment just in case
    if not ci.index.equals(idx):
        ci = ci.reindex(idx)
    return ci

def fit_ols(df, y, x, cov="classic", cluster_col=None):
    """
    Fit OLS y ~ const + x and return tidy table with beta, se, 95% CI, p-value, N, R2.
    cov in {'classic','HC3','cluster'}.
    """
    X = sm.add_constant(df[x])
    y_vec = df[y].to_numpy()
    base = sm.OLS(y_vec, X).fit()

    if cov == "classic":
        res = base
        note = "SE: classic"
    elif cov == "HC3":
        res = base.get_robustcov_results(cov_type="HC3")
        note = "SE: HC3 (robust)"
    elif cov == "cluster":
        if (cluster_col is None) or (cluster_col not in df.columns):
            return None, "Cluster column not available; skipped."
        res = base.get_robustcov_results(
            cov_type="cluster", groups=df[cluster_col], use_t=True
        )
        note = f"SE: clustered by {cluster_col} (t-based)"
    else:
        raise ValueError("cov must be one of: classic | HC3 | cluster")

    idx = _param_index(res)
    ci_df = _confint_df(res)

    # Build tidy table aligned on idx
    params = pd.Series(res.params, index=idx, name="beta")
    ses    = pd.Series(res.bse,    index=idx, name="se")
    pvals  = pd.Series(res.pvalues,index=idx, name="p_value")

    tbl = pd.concat([params, ses, pvals, ci_df], axis=1)

    # Keep only intercept and main regressor (whatever their names are)
    # common names: 'const' for intercept; x for regressor
    keep_names = []
    if "const" in tbl.index:
        keep_names.append("const")
    # try exact x; if not present (e.g., renamed), take the non-const term
    if X_MAIN in tbl.index:
        keep_names.append(X_MAIN)
    else:
        non_const = [n for n in tbl.index if n != "const"]
        if non_const:
            keep_names.append(non_const[0])  # safe fallback for 2-parameter model
    tbl = tbl.loc[keep_names]

    # Rename intercept row for presentation
    tbl = tbl.rename(index={"const":"Intercept", X_MAIN:X_MAIN})

    # meta
    tbl["N"]  = int(res.nobs)
    r2 = getattr(res, "rsquared_adj", None)
    if r2 is None:
        r2 = getattr(res, "rsquared", np.nan)
    tbl["R2"] = float(r2)
    tbl["covariance"] = note

    return tbl.reset_index().rename(columns={"index":"term"}), None

def make_clean_df(path):
    df = pd.read_csv(path)
    needed = [Y_LEVEL, X_MAIN]
    for c in needed:
        if c not in df.columns:
            raise ValueError(f"Column '{c}' not found in {path}")
    df = df.copy()
    df[Y_LOG] = np.log1p(df[Y_LEVEL])
    for col in [Y_LEVEL, Y_LOG, X_MAIN]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.dropna(subset=[Y_LEVEL, Y_LOG, X_MAIN]).reset_index(drop=True)
    return df

def run_spec(df, y, spec_label):
    rows = []
    for cov in ["classic", "HC3", "cluster"]:
        tbl, msg = fit_ols(df, y=y, x=X_MAIN, cov=cov, cluster_col=GROUP_CL)
        if tbl is None:
            print(f"[{spec_label}] {cov}: {msg}")
            continue
        tbl.insert(0, "model", spec_label)
        tbl.insert(1, "cov_type", cov)
        rows.append(tbl)
    if not rows:
        return None
    return pd.concat(rows, ignore_index=True)

# ---- Run everything ----
df = make_clean_df(IN_PATH)

table_level = run_spec(df, y=Y_LEVEL, spec_label="H1_level: sales ~ visitors")
table_log   = run_spec(df, y=Y_LOG,   spec_label="H1_log: log(sales) ~ visitors")

# Save & print
if table_level is not None:
    p = os.path.join(OUT_DIR, "baseline_level_CIs.csv")
    table_level.to_csv(p, index=False)
    print(f" Saved: {p}")
    print(table_level.to_string(index=False))

if table_log is not None:
    p = os.path.join(OUT_DIR, "baseline_log_CIs.csv")
    table_log.to_csv(p, index=False)
    print(f"\n Saved: {p}")
    print(table_log.to_string(index=False))

print("\n Notes:")
print("- 95% CIs via model.conf_int(0.05); no manual CI arithmetic.")
print("- HC3 = heteroskedasticity-robust; 'cluster' uses state if present.")
print("- Use these CSVs to overwrite the CI columns in Table 4.")

In [ ]:
## Block RF_CV — Random Forest with proper train/test + K-fold CV (report test only)

In [ ]:
# --- BLOCK RF_CV: Random Forest with holdout test + K-fold CV (report test only) ---

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error

IN_PATH = "data/private/analysis/sales_mobility_merged.csv.gz"

# 1) Load observed sample (no interpolation)
df = pd.read_csv(IN_PATH)

# 2) Features and target
#    Use mobility features available in your merged file; adjust if you have more.
feat_cols = [
    "total_unique_visitors","total_visits",
    "total_repeat_visitors","total_new_visitors",
    "avg_dwell_time_mins"
]
df = df.dropna(subset=["total_sales"] + feat_cols).copy()

X = df[feat_cols].values
y = np.log1p(df["total_sales"].values)   # log target is more stable

# 3) Train/test split
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

# 4) Model
rf = RandomForestRegressor(
    n_estimators=600,
    max_depth=None,
    min_samples_leaf=2,
    n_jobs=-1,
    random_state=42
)

# 5) Fit on train, evaluate on test (report this!)
rf.fit(X_tr, y_tr)
y_hat = rf.predict(X_te)

test_r2  = r2_score(y_te, y_hat)
test_rmse = mean_squared_error(y_te, y_hat, squared=False)

print(" Random Forest (test-only metrics)")
print(f"Test R²  : {test_r2:.3f}")
print(f"Test RMSE: {test_rmse:.3f}")

# 6) K-fold cross-validation on the whole observed dataset (for stability)
cv = KFold(n_splits=5, shuffle=True, random_state=42)
cv_res = cross_validate(
    rf, X, y,
    scoring=("r2","neg_root_mean_squared_error"),
    cv=cv, n_jobs=-1, return_train_score=False
)

cv_r2  = cv_res["test_r2"]
cv_rmse = -cv_res["test_neg_root_mean_squared_error"]

print("\n 5-fold CV (out-of-fold)")
print(f"R²  mean ± sd : {cv_r2.mean():.3f} ± {cv_r2.std():.3f}")
print(f"RMSE mean ± sd: {cv_rmse.mean():.3f} ± {cv_rmse.std():.3f}")

# 7) (Optional) Feature importance for appendix
imp = pd.Series(rf.feature_importances_, index=feat_cols).sort_values(ascending=False)
print("\n Feature importance:")
print(imp.round(4))

In [ ]:
## Block XGB_CV — XGBoost with holdout test + K-fold CV

In [ ]:
# --- Install XGBoost (run once in your Jupyter environment) ---
import sys


In [ ]:
# --- BLOCK XGB_CV: XGBoost with holdout test + K-fold CV (report test only) ---

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.metrics import r2_score, mean_squared_error
from xgboost import XGBRegressor

IN_PATH = "data/private/analysis/sales_mobility_merged.csv.gz"

# 1) Load observed sample
df = pd.read_csv(IN_PATH)

# 2) Features and target
feat_cols = [
    "total_unique_visitors","total_visits",
    "total_repeat_visitors","total_new_visitors",
    "avg_dwell_time_mins"
]
df = df.dropna(subset=["total_sales"] + feat_cols).copy()

X = df[feat_cols].values
y = np.log1p(df["total_sales"].values)   # log target

# 3) Train/test split
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

# 4) Model: tuned to be reasonable but not overfit
xgb = XGBRegressor(
    n_estimators=800,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    n_jobs=-1,
    random_state=42
)

# 5) Fit on train, evaluate on test
xgb.fit(X_tr, y_tr)
y_hat = xgb.predict(X_te)

test_r2  = r2_score(y_te, y_hat)
test_rmse = mean_squared_error(y_te, y_hat, squared=False)

print(" XGBoost (test-only metrics)")
print(f"Test R²  : {test_r2:.3f}")
print(f"Test RMSE: {test_rmse:.3f}")

# 6) K-fold cross-validation on full dataset
cv = KFold(n_splits=5, shuffle=True, random_state=42)
cv_res = cross_validate(
    xgb, X, y,
    scoring=("r2","neg_root_mean_squared_error"),
    cv=cv, n_jobs=-1, return_train_score=False
)

cv_r2  = cv_res["test_r2"]
cv_rmse = -cv_res["test_neg_root_mean_squared_error"]

print("\n 5-fold CV (out-of-fold)")
print(f"R²  mean ± sd : {cv_r2.mean():.3f} ± {cv_r2.std():.3f}")
print(f"RMSE mean ± sd: {cv_rmse.mean():.3f} ± {cv_rmse.std():.3f}")

# 7) Feature importance
imp = pd.Series(xgb.feature_importances_, index=feat_cols).sort_values(ascending=False)
print("\n Feature importance:")
print(imp.round(4))

In [ ]:
##Block S3 — IDW Leave-One-Out Validation + Sensitivity

In [ ]:
# --- BLOCK S3: IDW Leave-One-Out Validation (Observed GEWI) ---

import pandas as pd
import numpy as np
from shapely.geometry import Point
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
import os

# --- Inputs ---
OBS_PATH  = "data/private/analysis/sales_mobility_gewi_expanded.csv.gz"
CENT_PATH = "data/private/analysis/gewi_centroids.csv.gz"

# --- ensure parent dir helper ---
def ensure_parent_dir(path: str):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)

# --- Load observed GEWI (N≈96) ---
obs = pd.read_csv(OBS_PATH)
centroids = pd.read_csv(CENT_PATH)

obs = obs.merge(centroids, on="CD_SETOR", how="inner")
obs = obs.dropna(subset=["GEWI_v2_pos","lat","lon"]).reset_index(drop=True)

print(f"Observed GEWI sectors: {len(obs)}")

# --- IDW function ---
def idw_predict(target_idx, df, col="GEWI_v2_pos", radius_km=2.0, p=2):
    tgt = df.iloc[target_idx]
    others = df.drop(index=target_idx)

    # haversine distance (km)
    lat1, lon1 = np.radians(tgt["lat"]), np.radians(tgt["lon"])
    lat2, lon2 = np.radians(others["lat"].values), np.radians(others["lon"].values)
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    d = 6371 * 2*np.arcsin(np.sqrt(a))

    mask = d <= radius_km
    if not mask.any():
        return np.nan

    w = 1 / (d[mask]**p + 1e-9)
    vals = others.loc[mask, col].values
    return np.sum(w * vals) / np.sum(w)

# --- Loop over parameter grid ---
results = []
param_grid = [(r,p) for r in [1,2,3] for p in [1,2]]

for radius,p in param_grid:
    preds, trues = [], []
    for i in range(len(obs)):
        y_true = obs.iloc[i]["GEWI_v2_pos"]
        y_pred = idw_predict(i, obs, col="GEWI_v2_pos", radius_km=radius, p=p)
        if not np.isnan(y_pred):
            preds.append(y_pred)
            trues.append(y_true)
    if len(preds) < 5:
        continue
    mae  = mean_absolute_error(trues,preds)
    rmse = mean_squared_error(trues,preds,squared=False)
    r2   = r2_score(trues,preds)
    results.append((radius,p,mae,rmse,r2))

# --- Results table ---
val_table = pd.DataFrame(results, columns=["radius_km","p","MAE","RMSE","R2"])
print("\nIDW Leave-One-Out Validation:")
print(val_table)

# --- Save table ---
csv_path = "data/private/analysis/tables/IDW_LOO_validation.csv"
ensure_parent_dir(csv_path)
val_table.to_csv(csv_path,index=False)
print(f" Saved validation table → {csv_path}")

# --- Scatter plot for baseline (2km, p=2) ---
preds, trues = [], []
for i in range(len(obs)):
    y_true = obs.iloc[i]["GEWI_v2_pos"]
    y_pred = idw_predict(i, obs, col="GEWI_v2_pos", radius_km=2, p=2)
    if not np.isnan(y_pred):
        preds.append(y_pred)
        trues.append(y_true)

plt.figure(figsize=(6,6))
plt.scatter(trues,preds,alpha=0.7)
lims = [min(trues+preds), max(trues+preds)]
plt.plot(lims,lims,"r--")
plt.xlabel("Observed GEWI_v2_pos")
plt.ylabel("Predicted GEWI_v2_pos (IDW LOO, 2km p=2)")
plt.title("IDW Leave-One-Out Validation")

png_path = "data/private/analysis/figures/IDW_LOO_scatter.png"
ensure_parent_dir(png_path)
plt.savefig(png_path,dpi=300)
plt.close()
print(f" Saved scatter plot → {png_path}")

## Flexible-model support diagnostic

This descriptive check examines overlap between census tracts in the highest quartile of visit volume and the remaining tracts. The fitted probabilities summarize empirical support for this grouping and are not interpreted as propensity scores, treatment assignment, or evidence of causality.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

input_path = "data/private/analysis/sales_mobility_merged.csv.gz"
output_dir = "data/private/analysis"
figure_dir = os.path.join(output_dir, "figures")
table_dir = os.path.join(output_dir, "tables")
for directory in [figure_dir, table_dir]:
    os.makedirs(directory, exist_ok=True)

support_df = pd.read_csv(input_path)
if "total_visits" not in support_df.columns:
    raise ValueError("Column 'total_visits' not found in the analytical data.")

visit_q75 = support_df["total_visits"].quantile(0.75)
support_df["high_mobility_group"] = (
    support_df["total_visits"] >= visit_q75
).astype(int)

candidate_covariates = [
    "total_unique_visitors",
    "total_new_visitors",
    "total_repeat_visitors",
    "avg_dwell_time_mins",
    "GEWI_v2_pos",
    "GEWI_v2_neg",
    "GEWI_pos",
    "GEWI_neg",
]
covariates = [column for column in candidate_covariates if column in support_df.columns]
diagnostic = support_df[["high_mobility_group"] + covariates].dropna().copy()

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("logit", LogisticRegression(max_iter=5000, solver="lbfgs")),
])
pipeline.fit(diagnostic[covariates], diagnostic["high_mobility_group"])
diagnostic["support_score"] = pipeline.predict_proba(
    diagnostic[covariates]
)[:, 1]

high_scores = diagnostic.loc[
    diagnostic["high_mobility_group"] == 1, "support_score"
]
lower_scores = diagnostic.loc[
    diagnostic["high_mobility_group"] == 0, "support_score"
]

def summarize_group(series):
    return pd.Series({
        "n": len(series),
        "mean": series.mean(),
        "std": series.std(),
        "min": series.min(),
        "p01": series.quantile(0.01),
        "p05": series.quantile(0.05),
        "p10": series.quantile(0.10),
        "p90": series.quantile(0.90),
        "p95": series.quantile(0.95),
        "p99": series.quantile(0.99),
        "max": series.max(),
    })

score_summary = pd.concat(
    [summarize_group(high_scores), summarize_group(lower_scores)], axis=1
)
score_summary.columns = ["highest_visit_quartile", "remaining_tracts"]
score_summary.to_csv(
    os.path.join(table_dir, "support_score_summary_total_visits.csv")
)

lower_bound = max(high_scores.min(), lower_scores.min())
upper_bound = min(high_scores.max(), lower_scores.max())
share_in_support = diagnostic["support_score"].between(
    lower_bound, upper_bound
).mean()
pd.DataFrame({
    "lower_common_support": [lower_bound],
    "upper_common_support": [upper_bound],
    "share_in_common_support": [share_in_support],
}).to_csv(
    os.path.join(table_dir, "common_support_total_visits.csv"), index=False
)

def standardized_mean_difference(high_values, lower_values):
    pooled_sd = np.sqrt(
        (np.var(high_values, ddof=1) + np.var(lower_values, ddof=1)) / 2.0
    )
    if pooled_sd == 0:
        return 0.0
    return (np.mean(high_values) - np.mean(lower_values)) / pooled_sd

balance_rows = []
for column in covariates:
    high_values = diagnostic.loc[
        diagnostic["high_mobility_group"] == 1, column
    ].to_numpy()
    lower_values = diagnostic.loc[
        diagnostic["high_mobility_group"] == 0, column
    ].to_numpy()
    balance_rows.append({
        "covariate": column,
        "mean_highest_visit_quartile": np.mean(high_values),
        "mean_remaining_tracts": np.mean(lower_values),
        "standardized_mean_difference": standardized_mean_difference(
            high_values, lower_values
        ),
    })
pd.DataFrame(balance_rows).to_csv(
    os.path.join(table_dir, "covariate_balance_total_visits.csv"), index=False
)

bins = np.linspace(0, 1, 30)
plt.figure(figsize=(8, 6))
plt.hist(
    lower_scores,
    bins=bins,
    alpha=0.6,
    density=True,
    label="Remaining tracts",
    edgecolor="white",
)
plt.hist(
    high_scores,
    bins=bins,
    alpha=0.6,
    density=True,
    label="Highest visit-volume quartile",
    edgecolor="white",
)
plt.axvline(lower_bound, color="red", linestyle="--")
plt.axvline(upper_bound, color="red", linestyle="--")
plt.title("Empirical Support Across Visit-Volume Groups")
plt.xlabel("Estimated group-membership score")
plt.ylabel("Density")
plt.legend()
plt.savefig(
    os.path.join(figure_dir, "support_overlap_total_visits.png"),
    dpi=300,
    bbox_inches="tight",
)
plt.close()

print(score_summary.round(3))
print(
    f"Common support: [{lower_bound:.3f}, {upper_bound:.3f}] | "
    f"share in support = {share_in_support:.1%}"
)
